# D4 - Type Hints & Pydantic for Configuration

## Objective
Demonstrate application configuration management using Python type hints, Pydantic `BaseModel` / `BaseSettings`, field validation, path validation, range validation, enumerations, and custom domain error handling.

## Concepts Covered
- **Type Hints & Generic Types**: Strong typing with `str`, `int`, `float`, `Path`, `list[str]`.
- **Enumerations (`Enum`)**: Restricting options using `ExecutionMode` and `ExecutionDevice`.
- **Pydantic Validation**: `BaseModel` field validation via `@field_validator`.
- **Dataclass Comparison**: Demonstrating that standard `@dataclass` does not enforce runtime validation.
- **Fail-Fast Error Handling**: Converting Pydantic `ValidationError` into domain `ConfigError` via `load_pipeline_config`.

## Project Implementation
The configuration system resides in `app/config.py`:
- `ExecutionMode` & `ExecutionDevice`: Enums for pipeline mode and hardware target.
- `ImageSize`: Pydantic model validating positive dimensions.
- `PipelineConfig`: Pydantic model with validators for existing `input_path`, positive `batch_size`, non-empty `feature_columns`, and `confidence_threshold` range ($0.0 \le c \le 1.0$).
- `load_pipeline_config`: Helper converting Pydantic `ValidationError` into domain `ConfigError`.
- `DataclassPipelineConfig`: Dataclass showing lack of runtime enforcement.

## Demonstration
Below, we demonstrate valid configuration instantiation, dataclass comparison, and triggering all project validation error cases.

In [1]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.config import (
    PipelineConfig,
    load_pipeline_config,
    ExecutionMode,
    ExecutionDevice,
    ImageSize,
    DataclassPipelineConfig,
    ConfigError
)
from pydantic import ValidationError

# 1. Valid Configuration Instance
config = load_pipeline_config(
    input_path=root_dir,
    batch_size=64,
    mode=ExecutionMode.INFERENCE,
    device=ExecutionDevice.CPU,
    confidence_threshold=0.85
)

print("Valid PipelineConfig:")
print(f" - input_path: {config.input_path}")
print(f" - batch_size: {config.batch_size}")
print(f" - mode: {config.mode}")
print(f" - device: {config.device}")
print(f" - confidence_threshold: {config.confidence_threshold}")
print(f" - image_size: {config.image_size.width}x{config.image_size.height}")

Valid PipelineConfig:
 - input_path: C:\Users\Maha Monisha\OneDrive\Desktop\Triton Internship\task-management-api
 - batch_size: 64
 - mode: ExecutionMode.INFERENCE
 - device: ExecutionDevice.CPU
 - confidence_threshold: 0.85
 - image_size: 224x224


In [2]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.config import DataclassPipelineConfig

# 2. Dataclass Comparison (Lack of Runtime Validation)
dc_config = DataclassPipelineConfig(
    input_path="invalid/non_existent/path",
    batch_size=-999,
    confidence_threshold=99.9
)
print("Standard Dataclass instantiated without runtime validation:")
print(f" - input_path: {dc_config.input_path}")
print(f" - batch_size: {dc_config.batch_size}")
print(f" - confidence_threshold: {dc_config.confidence_threshold}")

Standard Dataclass instantiated without runtime validation:
 - input_path: invalid/non_existent/path
 - batch_size: -999
 - confidence_threshold: 99.9


In [3]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.config import load_pipeline_config, ConfigError

# 3. Invalid Case: Non-existent input_path
print("Testing Non-existent input_path...")
try:
    load_pipeline_config(input_path=Path("non_existent_dir_12345"))
except ConfigError as e:
    print(f"Caught Domain ConfigError:\n {e}")
    print(f"Underlying Cause: {type(e.__cause__).__name__}")

Testing Non-existent input_path...
Caught Domain ConfigError:
 PipelineConfig: configuration initialization failed; invalid arguments provided.
Underlying Cause: ValidationError


In [4]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.config import load_pipeline_config, ConfigError

# 4. Invalid Case: Invalid batch_size (<= 0)
print("Testing Invalid batch_size (-10)...")
try:
    load_pipeline_config(input_path=root_dir, batch_size=-10)
except ConfigError as e:
    print(f"Caught Domain ConfigError:\n {e}")

Testing Invalid batch_size (-10)...
Caught Domain ConfigError:
 PipelineConfig: configuration initialization failed; invalid arguments provided.


In [5]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.config import load_pipeline_config, ConfigError

# 5. Invalid Case: Invalid confidence_threshold (> 1.0)
print("Testing Invalid confidence_threshold (2.5)...")
try:
    load_pipeline_config(input_path=root_dir, confidence_threshold=2.5)
except ConfigError as e:
    print(f"Caught Domain ConfigError:\n {e}")

Testing Invalid confidence_threshold (2.5)...
Caught Domain ConfigError:
 PipelineConfig: configuration initialization failed; invalid arguments provided.


In [6]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from app.config import load_pipeline_config, ConfigError

# 6. Invalid Case: Invalid Enum & Incorrect Type
print("Testing Invalid Enum Value ('INVALID_MODE')...")
try:
    load_pipeline_config(input_path=root_dir, mode="INVALID_MODE")
except ConfigError as e:
    print(f"Caught Domain ConfigError:\n {e}")

Testing Invalid Enum Value ('INVALID_MODE')...
Caught Domain ConfigError:
 PipelineConfig: configuration initialization failed; invalid arguments provided.


## Actual Output
The code cells confirm:
1. Successful initialization of valid `PipelineConfig`.
2. Demonstration that standard `@dataclass` allows invalid data without validation.
3. Interception of non-existent paths, invalid batch sizes, out-of-range thresholds, and invalid enums, raising domain `ConfigError` wrapping Pydantic `ValidationError`.

## Key Observations
- Pydantic models validate data at runtime, whereas standard Python dataclasses only provide static type hints.
- Wrapping `ValidationError` into domain `ConfigError` decouples application logic from external validation library details.
- Configuration validation fails fast during initialization.

## Conclusion
Using Pydantic with type hints and domain exception wrapping provides robust, fail-fast configuration management for production applications.